# 02 Prompt Baselines

Run baseline zero-shot and few-shot prompting methods on the structured Synthea task.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import SEED, SYNTHEA_DIR
from src.data_prep import build_patient_dataset, filter_top_labels
from src.retrieval import build_retrieved_few_shot_context
from src.prompts import prompt_zero_shot_ranked, prompt_few_shot_ranked
from src.llm_runner import run_openai
from src.parsing import parse_ranked_output

patients_df = pd.read_csv(SYNTHEA_DIR / "patients.csv")
conditions_df = pd.read_csv(SYNTHEA_DIR / "conditions.csv")
df = build_patient_dataset(patients_df, conditions_df)
df, LABEL_SPACE = filter_top_labels(df, top_n=4)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["diagnosis"],
)

row = test_df.reset_index(drop=True).iloc[0]
few_shot_context = build_retrieved_few_shot_context(row, train_df.reset_index(drop=True), n=4)

zero_shot_prompt = prompt_zero_shot_ranked(row, LABEL_SPACE)
few_shot_prompt = prompt_few_shot_ranked(row, few_shot_context, LABEL_SPACE)

zero_shot_raw = run_openai(zero_shot_prompt)
few_shot_raw = run_openai(few_shot_prompt)

zero_shot_parsed = parse_ranked_output(zero_shot_raw, LABEL_SPACE)
few_shot_parsed = parse_ranked_output(few_shot_raw, LABEL_SPACE)

zero_shot_parsed, few_shot_parsed
